In [ ]:

# import vari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb


In [ ]:
# 100k righe occupano ~140mb di ram, tutto il file è decisamente troppo massiccio
# sample = pd.read_csv("./data/data.csv" , nrows=100000)
# sample.info(memory_usage="deep")

samplesize = 0.05

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ genera sample

# rng = np.random.default_rng(42)
# # quindi leggiamo il file a chunk di 100k righe 
# chunks = pd.read_csv("./data/data.csv", chunksize=100_000)
# # e per ogni chunk ne tiriamo fuori un pezzetto casuale, di grandezza un ventesimo, e concateniamo tutto
# sample = pd.concat(chunk.sample(frac=samplesize, random_state=rng) for chunk in chunks)
# # e poi salviamo in un pkl per non dover rivare tutto il giro ogni volta
# sample.to_pickle(f"./data/sample_{samplesize}.pkl")

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ carica sample
sample = pd.read_pickle(f"./data/sample_{samplesize}.pkl")

dataset = sample
dataset.info()
# origDataset = dataset.copy()
origFeatures = pd.DataFrame(index=dataset.index) # hold your reference and prepare to be sideffected
normalizationOutcome = pd.DataFrame(index=dataset.index)

In [ ]:
# funzioncine utili

# per rendere idempotenti i rerun delle celle e modificare inplace le feature senza perdersi gli originali
def backupFeature(featureName):
    if featureName not in origFeatures:
        origFeatures[featureName] = dataset[featureName]
    return origFeatures[featureName]

def featurePeek(toTable=False):

    valuesPeek = pd.DataFrame({
        "dtype": dataset.dtypes,
        "dist_vals": dataset.nunique(),
        "n_miss": dataset.isna().sum(),
        "%_miss": (dataset.isna().sum() / len(dataset) * 100),
        "values": pd.Series({c: dataset[c].value_counts().head(10).index.tolist()
                        for c in dataset.columns}),
    })

    if toTable:
        # i caratteri arabi irritano overleaf, quindi quando una stringa non è latin-1, la sostituiamo con un placeholder
        filterLatin = lambda s: str(s) if str(s).encode("latin-1", "ignore").decode("latin-1") == str(s) else "<non-latin-string>"
        valuesPeek["values"] = valuesPeek["values"].apply(lambda v: ", ".join(map(filterLatin, v)))

        # e poi, siccome overleaf regge male anche le sbrodolate, tagliamo la colonna values a un numero ragionevole di caratteri
        n = 20
        valuesPeek["values"] = valuesPeek["values"].apply(lambda s: s if len(s) <= n else s[:n] + "...")

        valuesPeek.to_latex("tables/feature_peek.tex", escape=True, float_format="%.1f",
                    column_format="llrrrl")
    return valuesPeek

In [ ]:
featurePeek()

# DATA UNDERSTANDING AND CLEANING


In [ ]:
# prima di tutto, assumendo che i postid siano univoci, ne abbiamo duplicati?
dataset["postid"].duplicated().sum()
dataset[dataset["postid"].duplicated(keep=False)].sort_values("postid")
# sì, ce ne sono. Sembra cambiare la country, è così?
dup = dataset.duplicated(subset=["postid", "ioCountry"], keep=False)
dataset[dup].sort_values(["postid", "ioCountry"])
# anche raggruppando per country ci sono comunque duplicati, conviene fermarsi qui per ora, 
# e fare un passaggio di normalizzazione

In [ ]:
# salviamo le colonne originali
fgc = backupFeature("following_count")
frc = backupFeature("follower_count")

# following e follower count hanno valori letterali, vediamo quali sono
fgcNumeric = pd.to_numeric(fgc, errors="coerce")

fgcInvalid = fgc[fgcNumeric.isna() & fgc.notna()]
# fgcInvalid.value_counts()
# ci sono sia valori proprio mancanti "nan-ish" che valori semplicemente mal formattati "ten"
# cosa fare? Prepariamo una funzione di mappatura che prova a normalizzare il valore ed emette sia il valore
# normalizzato che la tipologia del dato

def numericalNormalize(value):
    kv = {"ten": 10, "none": 0} # nel mondo reale qui ci andrebbe un ahimè, llm, per ora lo teniamo statico
    k = str(value).strip().lower()
    if k in kv:
        return kv[k], True # normalizzabile e normalizzato
    return pd.NA, False # non normalizzabile

# sostituiamo la colonna con quella normalizzata e aggiungiamo quella ausiliaria
normResult = fgcInvalid.map(numericalNormalize)
fgcNorm = fgcNumeric.fillna(normResult.str[0]) # dove numeric ha nan e il valore è normalizzabile, lo setta
dataset["following_count"] = fgcNorm.astype("Int64")
normalizationOutcome["followingCount"] = normResult.str[1].astype("boolean")

# e facciamo lo stesso per follower count
frcNumeric = pd.to_numeric(frc, errors="coerce")

frcInvalid = frc[frcNumeric.isna() & frc.notna()]
# frcInvalid.value_counts()

normResult = frcInvalid.map(numericalNormalize)
frcNorm = frcNumeric.fillna(normResult.str[0]) 
dataset["follower_count"] = frcNorm.astype("Int64")
normalizationOutcome["follower_count"] = normResult.str[1].astype("boolean")



In [ ]:
# passiamo a is_control
# dataset["is_control"].value_counts()
# come sopra, ma con i booleani presi come stringa
isc = backupFeature("is_control")

def booleanNormalize(value):
    # se è già mancante o già bool non serve far nulla
    if pd.isna(value) or value in (True, False):
        return value, pd.NA
    kv = {"true": True, "false": False}
    k = str(value).strip().lower()
    if k in kv:
        return kv[k], True
    return pd.NA, False

normResult = isc.map(booleanNormalize)
dataset["is_control"] = normResult.str[0].astype("boolean")
normalizationOutcome["is_control"] = normResult.str[1].astype("boolean")

In [ ]:
# le marche temporali sono scritte come stringhe, ma a noi piacciono come datetime
pt = backupFeature("post_time")
act = backupFeature("account_creation_date")

dataset["post_time"] = pd.to_datetime(dataset["post_time"], format="mixed", errors="coerce")
# dataset["post_time"].isna().any()

dataset["account_creation_date"] = pd.to_datetime(dataset["account_creation_date"], format="mixed", errors="coerce")
# dataset["account_creation_date"].isna().any()

In [ ]:
featurePeek()

# DATA VISUALIZATION

In [ ]:

# Set up a grid of plots:
fig = plt.figure(figsize=(20, 10)) 
fig_dims = (4, 3)
fig.subplots_adjust(hspace=0.4, wspace=0.4)

plt.subplot2grid(fig_dims, (0, 0))
dataset["ioCountry"].value_counts().plot(kind='bar',title='numero di post per stato')

plt.subplot2grid(fig_dims, (0, 1))
# non è molto informativo vedere la distribuzione dei post nel tempo
ptc = dataset["post_time"]
buckets = pd.cut(ptc, bins=20).value_counts(sort=False)
# poi sistemiamo l'etichetta da precisione piena a giusto il giorno
buckets.index = [f"{b.left:%Y-%m-%d} – {b.right:%Y-%m-%d}" for b in buckets.index]
# e poi plottiamo
buckets.plot(kind="bar", title="periodo dei post")
# dataset["post_time_clean"].plot(kind="hist", title="periodo dei post", bins=20)

plt.subplot2grid(fig_dims, (0, 2))
# usare la lunghezza del testo è un po' grezzo, così come il numero di parole (possono cambiare col linguaggio),
# ma per farsi un'idea iniziale funziona. 

dataset["post_text"].str.len().plot(kind="hist", bins=30) 
# anche in questo caso ci sono evidentemente degli outlier molto lunghi e poco comuni


In [ ]:

# proviamo a farci un'idea di quanto i diversi paesi hanno avuto iops attive nello stesso periodo
# come prima usiamo dei bucket uniformi per tutti
buckets = pd.cut(dataset["post_time"], bins=20)
# calcoliamo la tabella di contingenza, giusto per avere in un asse i bucket, nell'altro le countries, e nelle celle
# il numero di post relativi 
tabella = pd.crosstab(buckets, dataset["ioCountry"])

tabella.index = [f"{b.left:%Y-%m-%d}" for b in tabella.index]
tabella.plot(figsize=(14, 6))